In [6]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter, MarkdownTextSplitter, CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_chroma import Chroma

In [7]:
from langchain_core.documents import Document 

import jsonlines
video_docs = []
file_path = "Doc.jsonl"
with jsonlines.open(file_path, mode='r') as reader:
    for doc_dict in reader:
        video_docs.append(Document(**doc_dict))

In [8]:
# for video in video_docs:
#     if 'V15' in video.page_content:
#         print(video)
#         print(len(video.page_content.split(" ")))

In [9]:
video_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 2000, chunk_overlap=500
)

video_splits = video_splitter.split_documents(video_docs)

model_embeddings_rag = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'batch_size': 1}
)
vector_db = Chroma.from_documents(
                        documents=video_splits,
                        embedding=model_embeddings_rag,
                        collection_name="video_vector_db",
                        persist_directory="./chroma_db"
                    )

retriever = vector_db.as_retriever()

In [10]:
# embeddings = HuggingFaceEmbeddings(
#     model_name="intfloat/multilingual-e5-base"
# )

# # 2. Load the existing database from disk
# vector_db = Chroma(
#     persist_directory="./chroma_db",
#     embedding_function=embeddings,
#     collection_name="video_vector_db"
# )

# retriever = vector_db.as_retriever()

In [12]:
def return_video_link(docs):
    return set([doc.metadata['source'] for doc in docs])

video_chain = (
    retriever
    | return_video_link
)
chain = RunnableParallel(
    links=video_chain,
    question=RunnablePassthrough()
)

chain.invoke("como configurar sunat")

{'links': {'https://youtu.be/MHyybBZNm78',
  'https://youtu.be/kXgXcKjvc9Y',
  'https://youtu.be/wqziZeKGoqY',
  'https://youtu.be/yPoSX7pOvqA'},
 'question': 'como configurar sunat'}